In [1]:
# ===== NeurIPS Open Polymer Prediction 2025 — Level 1 Upgrade with Auto-Selection =====
# - Baseline: TF-IDF (char 2–5) + Ridge
# - Upgrade: + SMILES stats (counts) and optional XGBoost (MAE) if available
# - Per-target model selection by 5-fold CV (approx wMAE). Guarantees no regression.

import os, glob, re, gc, math
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from scipy import sparse

RANDOM_STATE = 42
N_SPLITS = 5
TARGETS = ["Tg","FFV","Tc","Density","Rg"]

# Try optional boosters (safe if installed in Kaggle image)
HAS_XGB = False
try:
    import xgboost as xgb
    HAS_XGB = True
except Exception:
    HAS_XGB = False

# ---------- Locate dataset ----------
candidates = [d for d in glob.glob("/kaggle/input/*")
              if any(k in d.lower() for k in ["polymer","neurips","open","opp"])]
DATA_DIR = None
for d in candidates:
    if all(os.path.exists(os.path.join(d, f)) for f in ["train.csv","test.csv","sample_submission.csv"]):
        DATA_DIR = d
        break
if DATA_DIR is None:
    DATA_DIR = "/kaggle/input/neurips-open-polymer-prediction-2025"
print("Using data dir:", DATA_DIR)

train = pd.read_csv(os.path.join(DATA_DIR,"train.csv"))
test  = pd.read_csv(os.path.join(DATA_DIR,"test.csv"))
sample = pd.read_csv(os.path.join(DATA_DIR,"sample_submission.csv"))

# ---------- Identify columns ----------
def find_smiles(df):
    for c in df.columns:
        if c.lower().strip() in ("smiles","smile","smiles_str","polymer_smiles"):
            return c
    raise KeyError("Could not find SMILES column in", df.columns.tolist())

SMI_COL = find_smiles(train)
assert SMI_COL == find_smiles(test)

def map_targets(df):
    m = {}
    for want in TARGETS:
        for c in df.columns:
            if c.lower() == want.lower():
                m[want] = c
                break
    return m

tmap = map_targets(train)
TARGETS_CAN = [t for t in TARGETS if t in tmap]
print("Targets present:", TARGETS_CAN)

# ---------- SMILES stats (cheap count features) ----------
# Counts that often help: length, parentheses, bonds, ring digits, atoms, aromatics, dots, percent (two-digit rings)
ATOM_TOKENS = ["Cl","Br","Si"]
SINGLE_CHARS = list("BCNOFPSI")  # capitals likely to appear standalone (B, C, N, O, F, P, S, I)
AROM_CHARS = list("cnospb")       # lowercase aromatics
BOND_CHARS = list("=#")           # double, triple
SYMS = list("()[]")               # branch/square brackets
EXTRA = [".","%"]                 # dot (fragments), % (two-digit rings)

RING_DIGITS = list("123456789")

def smiles_stats_vector(smiles: str):
    s = smiles
    L = len(s)

    # Count two-letter tokens first to avoid double-counting their letters
    counts = {}
    for tk in ATOM_TOKENS:
        counts[tk] = len(re.findall(tk, s))
        s = s.replace(tk, " ")  # mask to reduce double-counting

    # Now count single-character tokens on masked string
    for ch in SINGLE_CHARS:
        counts[ch] = s.count(ch)

    for ch in AROM_CHARS:
        counts[f"ar_{ch}"] = s.count(ch)

    for ch in BOND_CHARS + SYMS + EXTRA:
        counts[ch] = s.count(ch)

    # ring digits (on original)
    s_orig = smiles
    for d in RING_DIGITS:
        counts[f"ring_{d}"] = s_orig.count(d)

    # simple aggregates/ratios (guard zero)
    ring_total = sum(counts[f"ring_{d}"] for d in RING_DIGITS)
    aromatic_total = sum(counts[f"ar_{ch}"] for ch in AROM_CHARS)
    caps_total = sum(counts.get(ch,0) for ch in SINGLE_CHARS) + sum(counts[tk] for tk in ATOM_TOKENS)
    bonds_total = sum(counts[ch] for ch in BOND_CHARS)
    parens_total = counts["("] + counts[")"]

    feats = [
        L,
        ring_total,
        aromatic_total,
        caps_total,
        bonds_total,
        parens_total,
        counts["["] + counts["]"],
        counts["."],
        counts["%"],
        counts.get("Cl",0), counts.get("Br",0), counts.get("Si",0),
        counts.get("C",0), counts.get("N",0), counts.get("O",0),
        counts.get("F",0), counts.get("P",0), counts.get("S",0), counts.get("I",0),
    ]

    # ratios (normalized by length)
    def ratio(x): return (x / L) if L > 0 else 0.0
    feats += [
        ratio(ring_total), ratio(aromatic_total), ratio(caps_total),
        ratio(bonds_total), ratio(parens_total)
    ]
    return np.array(feats, dtype=np.float32)

STAT_NAMES = [
    "len","rings","arom","caps","bonds","parens","brackets","dots","pct",
    "Cl","Br","Si","C","N","O","F","P","S","I",
    "r_rings","r_arom","r_caps","r_bonds","r_parens"
]

def featurize_stats(smiles_list):
    M = len(smiles_list)
    X = np.zeros((M, len(STAT_NAMES)), dtype=np.float32)
    for i, smi in enumerate(smiles_list):
        X[i,:] = smiles_stats_vector(str(smi))
    return X

# ---------- TF-IDF config ----------
def make_tfidf():
    return TfidfVectorizer(
        analyzer="char",
        ngram_range=(2,5),
        min_df=2,
        max_features=200_000
    )

# ---------- Approximate wMAE (train-based weights) ----------
def compute_weights(y_df, cols):
    # n_i, r_i from train labels
    n = y_df[cols].notna().sum().astype(float)
    r = (y_df[cols].max() - y_df[cols].min()).astype(float).replace(0, 1.0)
    K = len(cols)
    inv_sqrt = 1/np.sqrt(n.clip(lower=1))
    norm = K * (inv_sqrt / inv_sqrt.sum())
    w = (1.0/r) * norm
    return w  # pd.Series indexed by cols

def approx_wmae(y_df, yhat_df, cols, weights):
    per_row = []
    for i in range(len(y_df)):
        s = 0.0
        for c in cols:
            yi = y_df.at[i,c]
            if pd.notna(yi):
                s += weights[c] * abs(yhat_df.at[i,c] - yi)
        per_row.append(s)
    denom = (y_df[cols].notna().sum(axis=1) > 0).sum()
    return float(np.sum(per_row) / max(denom,1))

# ---------- KFold ----------
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

# Precompute stats for all rows (fast)
train_smiles = train[SMI_COL].astype(str).values
test_smiles  = test[SMI_COL].astype(str).values
Xstats_train = featurize_stats(train_smiles)
Xstats_test  = featurize_stats(test_smiles)

# Prepare target df
y_full = train[[tmap[t] for t in TARGETS_CAN]].copy()
y_full.columns = TARGETS_CAN
weights = compute_weights(y_full, TARGETS_CAN)

# Storage for final chosen predictions
OOF_best = pd.DataFrame(index=train.index, columns=TARGETS_CAN, dtype=float)
PRED_best = pd.DataFrame({"id": test["id"].values})
for t in TARGETS_CAN: PRED_best[t] = 0.0

# For reporting baseline vs selected
report_rows = []

# Candidate configs per target
# Each function returns (oof_pred, test_pred, score_contribution)
def train_target_with_config(target, config_name):
    mask = ~pd.isna(train[tmap[target]])
    y = train.loc[mask, tmap[target]].values
    X_text = train_smiles[mask]
    Xs = Xstats_train[mask]
    Xt_text_test = test_smiles
    Xs_test = Xstats_test

    oof = np.zeros(mask.sum(), dtype=np.float32)
    pred_test = np.zeros(len(test), dtype=np.float32)

    for fold, (tr_idx, va_idx) in enumerate(kf.split(X_text), 1):
        X_tr_text, X_va_text = X_text[tr_idx], X_text[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]
        tfidf = make_tfidf()
        Xtr_tfidf = tfidf.fit_transform(X_tr_text)
        Xva_tfidf = tfidf.transform(X_va_text)
        Xtst_tfidf = tfidf.transform(Xt_text_test)

        if config_name == "ridge_tfidf":
            Xtr = Xtr_tfidf
            Xva = Xva_tfidf
            Xtst = Xtst_tfidf
            model = Ridge(alpha=5.0, random_state=RANDOM_STATE)
            model.fit(Xtr, y_tr)
            oof[va_idx] = model.predict(Xva)
            pred_test += model.predict(Xtst) / N_SPLITS

        elif config_name == "ridge_tfidf_stats":
            # Combine sparse tf-idf with dense stats via hstack
            Xtr = sparse.hstack([Xtr_tfidf, sparse.csr_matrix(Xs[tr_idx])]).tocsr()
            Xva = sparse.hstack([Xva_tfidf, sparse.csr_matrix(Xs[va_idx])]).tocsr()
            Xtst = sparse.hstack([Xtst_tfidf, sparse.csr_matrix(Xs_test)]).tocsr()
            model = Ridge(alpha=5.0, random_state=RANDOM_STATE)
            model.fit(Xtr, y_tr)
            oof[va_idx] = model.predict(Xva)
            pred_test += model.predict(Xtst) / N_SPLITS

        elif config_name == "xgb_tfidf_stats":
            if not HAS_XGB:
                raise RuntimeError("XGBoost not available")
            Xtr = sparse.hstack([Xtr_tfidf, sparse.csr_matrix(Xs[tr_idx])]).tocsr()
            Xva = sparse.hstack([Xva_tfidf, sparse.csr_matrix(Xs[va_idx])]).tocsr()
            Xtst = sparse.hstack([Xtst_tfidf, sparse.csr_matrix(Xs_test)]).tocsr()
            model = xgb.XGBRegressor(
                n_estimators=400,
                learning_rate=0.05,
                max_depth=6,
                subsample=0.8,
                colsample_bytree=0.5,
                objective="reg:absoluteerror",
                tree_method="hist",
                random_state=RANDOM_STATE,
                n_jobs=-1,
                reg_lambda=1.0,
            )
            model.fit(Xtr, y_tr, eval_set=[(Xva, y_va)],
                      verbose=False, early_stopping_rounds=50)
            oof[va_idx] = model.predict(Xva)
            pred_test += model.predict(Xtst) / N_SPLITS

        else:
            raise ValueError("Unknown config")

        # cleanup
        del Xtr_tfidf, Xva_tfidf, Xtst_tfidf
        gc.collect()

    # Weighted contribution for this target only
    mae = mean_absolute_error(y, oof)
    w_contrib = float(weights[target] * mae)
    return oof, pred_test, w_contrib, mae

# Train & select per target
CONFIGS = ["ridge_tfidf", "ridge_tfidf_stats"] + (["xgb_tfidf_stats"] if HAS_XGB else [])
print("Configs to try:", CONFIGS)

for target in TARGETS_CAN:
    print(f"\n=== {target} — model selection over {len(CONFIGS)} configs ===")
    best = None
    results = []
    per_config_store = {}

    for cfg in CONFIGS:
        try:
            oof, pred_t, w_contrib, mae = train_target_with_config(target, cfg)
            results.append((cfg, w_contrib, mae))
            per_config_store[cfg] = (oof, pred_t, w_contrib, mae)
            print(f"  {cfg:17s} -> w*MAE={w_contrib:.6f}  (MAE={mae:.6f})")
        except Exception as e:
            print(f"  {cfg:17s} -> SKIPPED ({e})")

    # Choose best by weighted contribution (lower is better)
    if not results:
        raise RuntimeError(f"No configs succeeded for {target}")
    results.sort(key=lambda x: x[1])
    best_cfg, best_w, best_mae = results[0][0], results[0][1], results[0][2]
    print(f"  -> Selected: {best_cfg}  (w*MAE={best_w:.6f}, MAE={best_mae:.6f})")

    # Also compute baseline (ridge_tfidf) for reporting delta
    base_w = None; base_mae = None
    if "ridge_tfidf" in per_config_store:
        base_mae = per_config_store["ridge_tfidf"][3]
        base_w = per_config_store["ridge_tfidf"][2]

    # Write chosen preds into final storage
    oof_best, pred_best = per_config_store[best_cfg][0], per_config_store[best_cfg][1]
    mask = ~pd.isna(train[tmap[target]])
    OOF_best.loc[mask, target] = oof_best
    PRED_best[target] = pred_best

    report_rows.append({
        "target": target,
        "baseline_MAE": base_mae,
        "selected_cfg": best_cfg,
        "selected_MAE": best_mae,
        "MAE_delta": None if base_mae is None else (base_mae - best_mae),
        "baseline_w*MAE": base_w,
        "selected_w*MAE": best_w,
        "w*MAE_delta": None if base_w is None else (base_w - best_w),
    })

# Overall approx wMAE (baseline vs selected), if baseline available for all
report_df = pd.DataFrame(report_rows)
print("\nPer-target selection summary:")
print(report_df)

# Compute overall approx wMAE for the selected set
sel_wmae = approx_wmae(y_full, OOF_best[TARGETS_CAN], TARGETS_CAN, weights)
print("\nApprox overall wMAE (selected per-target models):", round(sel_wmae, 6))
print("Weights used:", weights.to_dict())

# If every target had a ridge_tfidf run, compute its overall too for comparison
try:
    # Rebuild a pseudo-OOF of pure baseline from stored pieces
    OOF_baseline = pd.DataFrame(index=train.index, columns=TARGETS_CAN, dtype=float)
    for r in report_rows:
        if r["baseline_MAE"] is None:
            raise RuntimeError("No full baseline")
    # If we reach here, we can recompute ridge_tfidf OOF by re-training per target
    # (Light cost; but to save time, we can skip and just report per-target deltas)
    print("\n(Baseline per-target deltas are shown above; recomputing overall baseline is skipped for speed.)")
except Exception:
    pass

# ---------- Submission ----------
sub = pd.DataFrame({
    "id": test["id"].values,
    "Tg": PRED_best["Tg"] if "Tg" in PRED_best else 0.0,
    "FFV": PRED_best["FFV"] if "FFV" in PRED_best else 0.0,
    "Tc": PRED_best["Tc"] if "Tc" in PRED_best else 0.0,
    "Density": PRED_best["Density"] if "Density" in PRED_best else 0.0,
    "Rg": PRED_best["Rg"] if "Rg" in PRED_best else 0.0,
})
save_path = "/kaggle/working/submission.csv"
sub.to_csv(save_path, index=False)
print("\nSaved:", save_path)


Using data dir: /kaggle/input/neurips-open-polymer-prediction-2025
Targets present: ['Tg', 'FFV', 'Tc', 'Density', 'Rg']
Configs to try: ['ridge_tfidf', 'ridge_tfidf_stats', 'xgb_tfidf_stats']

=== Tg — model selection over 3 configs ===
  ridge_tfidf       -> SKIPPED (cg() got an unexpected keyword argument 'tol')
  ridge_tfidf_stats -> SKIPPED (cg() got an unexpected keyword argument 'tol')


/usr/local/lib/python3.11/dist-packages/xgboost/sklearn.py:889: UserWarning: `early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds` in constructor or`set_params` instead.
  warnings.warn(


  xgb_tfidf_stats   -> w*MAE=0.111198  (MAE=54.179949)
  -> Selected: xgb_tfidf_stats  (w*MAE=0.111198, MAE=54.179949)

=== FFV — model selection over 3 configs ===
  ridge_tfidf       -> SKIPPED (cg() got an unexpected keyword argument 'tol')
  ridge_tfidf_stats -> SKIPPED (cg() got an unexpected keyword argument 'tol')


/usr/local/lib/python3.11/dist-packages/xgboost/sklearn.py:889: UserWarning: `early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds` in constructor or`set_params` instead.
  warnings.warn(


  xgb_tfidf_stats   -> w*MAE=0.004660  (MAE=0.007469)
  -> Selected: xgb_tfidf_stats  (w*MAE=0.004660, MAE=0.007469)

=== Tc — model selection over 3 configs ===
  ridge_tfidf       -> SKIPPED (cg() got an unexpected keyword argument 'tol')
  ridge_tfidf_stats -> SKIPPED (cg() got an unexpected keyword argument 'tol')


...

  xgb_tfidf_stats   -> w*MAE=0.060961  (MAE=0.027460)
  -> Selected: xgb_tfidf_stats  (w*MAE=0.060961, MAE=0.027460)

=== Density — model selection over 3 configs ===
  ridge_tfidf       -> SKIPPED (cg() got an unexpected keyword argument 'tol')
  ridge_tfidf_stats -> SKIPPED (cg() got an unexpected keyword argument 'tol')


/usr/local/lib/python3.11/dist-packages/xgboost/sklearn.py:889: UserWarning: `early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds` in constructor or`set_params` instead.
  warnings.warn(


  xgb_tfidf_stats   -> w*MAE=0.046542  (MAE=0.043739)
  -> Selected: xgb_tfidf_stats  (w*MAE=0.046542, MAE=0.043739)

=== Rg — model selection over 3 configs ===
  ridge_tfidf       -> SKIPPED (cg() got an unexpected keyword argument 'tol')
  ridge_tfidf_stats -> SKIPPED (cg() got an unexpected keyword argument 'tol')


/usr/local/lib/python3.11/dist-packages/xgboost/sklearn.py:889: UserWarning: `early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds` in constructor or`set_params` instead.
  warnings.warn(


  xgb_tfidf_stats   -> w*MAE=0.072784  (MAE=1.563295)
  -> Selected: xgb_tfidf_stats  (w*MAE=0.072784, MAE=1.563295)

Per-target selection summary:
    target baseline_MAE     selected_cfg  selected_MAE MAE_delta  \
0       Tg         None  xgb_tfidf_stats     54.179949      None   
1      FFV         None  xgb_tfidf_stats      0.007469      None   
2       Tc         None  xgb_tfidf_stats      0.027460      None   
3  Density         None  xgb_tfidf_stats      0.043739      None   
4       Rg         None  xgb_tfidf_stats      1.563295      None   

  baseline_w*MAE  selected_w*MAE w*MAE_delta  
0           None        0.111198        None  
1           None        0.004660        None  
2           None        0.060961        None  
3           None        0.046542        None  
4           None        0.072784        None  

Approx overall wMAE (selected per-target models): 0.026054
Weights used: {'Tg': 0.00205237749659811, 'FFV': 0.6239248659724905, 'Tc': 2.2199754352943795, 'Densi